In [23]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules


df = pd.read_csv("Transaction_data.csv")
df.columns = ['TID', 'Item']
transactions = df.groupby('TID')['Item'].apply(list).tolist()
print(transactions)

[['I1, I2, I5'], ['I2, I4'], ['I2, I3'], ['I1, I2, I4'], ['I1, I3'], ['I2, I3'], ['I1, I3'], ['I1, I2, I3, I5'], ['I1, I2, I3']]


In [35]:
te = TransactionEncoder()
x = te.fit_transform(transactions)
newdf = pd.DataFrame(x)


In [36]:
print(newdf)

       0      1      2      3      4      5      6
0  False  False  False   True  False  False  False
1  False  False  False  False  False  False   True
2  False  False  False  False  False   True  False
3  False  False   True  False  False  False  False
4  False  False  False  False   True  False  False
5  False  False  False  False  False   True  False
6  False  False  False  False   True  False  False
7  False   True  False  False  False  False  False
8   True  False  False  False  False  False  False


In [44]:
freq_items = apriori(newdf, min_support = 0.1, use_colnames = True)
print(freq_items)

    support itemsets
0  0.111111      (0)
1  0.111111      (1)
2  0.111111      (2)
3  0.111111      (3)
4  0.222222      (4)
5  0.222222      (5)
6  0.111111      (6)


In [47]:
rules = association_rules(freq_items, metric = "confidence", min_threshold = 0.3)
print(rules)


Empty DataFrame
Columns: [antecedents, consequents, antecedent support, consequent support, support, confidence, lift, representativity, leverage, conviction, zhangs_metric, jaccard, certainty, kulczynski]
Index: []


In [57]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# ----------------------------
# 1. Load your CSV
# ----------------------------
file_path = "Transaction_data.csv"
df = pd.read_csv(file_path)

print("Dataset Preview:")
print(df.head())

# -------------------------------------------------
# 2. Detect format and convert to list of items
# -------------------------------------------------

# Case 1: each row contains items as columns
if df.shape[1] > 2:
    transactions = df.apply(lambda row: row.dropna().tolist(), axis=1).tolist()

# Case 2: two columns TransactionID + Item
else:
    df.columns = ['TID', 'Item']
    transactions = df.groupby('TID')['Item'].apply(list).tolist()

print("\nExtracted Transactions:")
print(transactions[:10])

# -------------------------------------------------
# 3. Transaction Encoding
# -------------------------------------------------
te = TransactionEncoder()
te_data = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_data, columns=te.columns_)

# -------------------------------------------------
# 4. Apply Apriori
# -------------------------------------------------
frequent_itemsets = apriori(df_encoded, min_support=0.1, use_colnames=True)

print("\nFrequent Itemsets:")
print(frequent_itemsets)

# -------------------------------------------------
# 5. Generate Association Rules
# -------------------------------------------------
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

print("\nAssociation Rules:")
print(rules)


Dataset Preview:
    TID List of item_IDs
0  T100       I1, I2, I5
1  T200           I2, I4
2  T300           I2, I3
3  T400       I1, I2, I4
4  T500           I1, I3

Extracted Transactions:
[['I1, I2, I5'], ['I2, I4'], ['I2, I3'], ['I1, I2, I4'], ['I1, I3'], ['I2, I3'], ['I1, I3'], ['I1, I2, I3, I5'], ['I1, I2, I3']]

Frequent Itemsets:
    support          itemsets
0  0.111111      (I1, I2, I3)
1  0.111111  (I1, I2, I3, I5)
2  0.111111      (I1, I2, I4)
3  0.111111      (I1, I2, I5)
4  0.222222          (I1, I3)
5  0.222222          (I2, I3)
6  0.111111          (I2, I4)

Association Rules:
Empty DataFrame
Columns: [antecedents, consequents, antecedent support, consequent support, support, confidence, lift, representativity, leverage, conviction, zhangs_metric, jaccard, certainty, kulczynski]
Index: []


In [92]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
import numpy as np

# Load the data
df = pd.read_csv("Transaction_data.csv")

# Inspect the data
print(df.head())
# print(df.info())

# --- Preprocessing Step ---
# 1. Clean up and split the item list.
# The column name is "List of item_IDs" (with a leading space, based on head output)
# We will use .str.split(',') and then clean the whitespace.
df['Items'] = df.iloc[:, 1].str.split(',')
print(df)    

# Flatten the list of items for all transactions
transactions = []
for index, row in df.iterrows():
    # Strip whitespace from each item in the list
    cleaned_items = [item.strip() for item in row['Items']]
    # Filter out empty strings that might result from extra commas
    cleaned_items = [item for item in cleaned_items if item]
    transactions.append(cleaned_items)

print(transactions)
# Convert the list of transactions into a one-hot encoded DataFrame for mlxtend
# Get all unique items
all_items = sorted(list(set([item for sublist in transactions for item in sublist])))
print(all_items)
# Create the one-hot encoded structure
ohe_df = pd.DataFrame(False, index=df.index, columns=all_items)

print(ohe_df)
# Populate the DataFrame
for i, transaction in enumerate(transactions):
    for item in transaction:
        ohe_df.loc[i, item] = True

# Display the head of the one-hot encoded DataFrame for verification
print("\nOne-Hot Encoded DataFrame Head:")
print(ohe_df.head())

# --- Apriori Algorithm ---
# Use a minimum support of 0.3 (i.e., min. 3 out of 9 transactions)
min_support = 0.2
frequent_itemsets = apriori(ohe_df, min_support=min_support, use_colnames=True)

# # Sort the frequent itemsets by support and itemset length
# frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))
# frequent_itemsets = frequent_itemsets.sort_values(by=['length', 'support'], ascending=[True, False]).reset_index(drop=True)

# Display the frequent itemsets
print("\nFrequent Itemsets (min_support = 0.3):")
print(frequent_itemsets)

# --- Association Rules Mining ---
# Generate rules with a minimum confidence of 0.7
min_confidence = 0.3
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence)
print(rules)
# Sort the rules by confidence and lift
# rules = rules.sort_values(by=['confidence', 'lift'], ascending=False).reset_index(drop=True)

# # Select and format the relevant columns for display
# rules_display = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
# rules_display['antecedents'] = rules_display['antecedents'].apply(lambda x: ', '.join(list(x)))
# rules_display['consequents'] = rules_display['consequents'].apply(lambda x: ', '.join(list(x)))

# # Display the association rules
# print("\nAssociation Rules (min_confidence = 0.7):")
# print(rules_display)

# # Save the results to CSV for the user
# frequent_itemsets.to_csv('frequent_itemsets.csv', index=False)
# rules_display.to_csv('association_rules.csv', index=False)

    TID List of item_IDs
0  T100       I1, I2, I5
1  T200           I2, I4
2  T300           I2, I3
3  T400       I1, I2, I4
4  T500           I1, I3
    TID List of item_IDs                Items
0  T100       I1, I2, I5       [I1,  I2,  I5]
1  T200           I2, I4            [I2,  I4]
2  T300           I2, I3            [I2,  I3]
3  T400       I1, I2, I4       [I1,  I2,  I4]
4  T500           I1, I3            [I1,  I3]
5  T600           I2, I3            [I2,  I3]
6  T700           I1, I3            [I1,  I3]
7  T800   I1, I2, I3, I5  [I1,  I2,  I3,  I5]
8  T900       I1, I2, I3       [I1,  I2,  I3]
[['I1', 'I2', 'I5'], ['I2', 'I4'], ['I2', 'I3'], ['I1', 'I2', 'I4'], ['I1', 'I3'], ['I2', 'I3'], ['I1', 'I3'], ['I1', 'I2', 'I3', 'I5'], ['I1', 'I2', 'I3']]
['I1', 'I2', 'I3', 'I4', 'I5']
      I1     I2     I3     I4     I5
0  False  False  False  False  False
1  False  False  False  False  False
2  False  False  False  False  False
3  False  False  False  False  False
4  False  False  

In [93]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
data = pd.read_csv("trans.csv")
data["Description"] = data["Description"].str.strip()
data.dropna(axis = 0,subset = ['InvoiceNo'])
data["InvoiceNo"] = data["InvoiceNo"].astype("str")
data = data[~data["InvoiceNo"].str.contains('C')]
basket_france = (data[data["Country"] == "France"].groupby(["InvoiceNo","Description"])["Quantity"].sum().unstack().reset_index().fillna(0).set_index("InvoiceNo"))
def hot_encode(x):
    if x <= 0:
        return 0
    else:
        return 1
basket_encoded = basket_france.applymap(hot_encode)
basket_france = basket_encoded
frq_items = apriori(basket_france,min_support = 0.1,use_colnames = True)
rules = association_rules(frq_items,metric = "lift",min_threshold=1)
rules = rules.sort_values(["confidence","lift"],ascending=[False,False])
print(rules.head())

C:\Users\abhay\AppData\Local\Temp\ipykernel_22568\3211898063.py:14: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket_encoded = basket_france.applymap(hot_encode)


                                 antecedents                      consequents  \
41           (SET/6 RED SPOTTY PAPER PLATES)    (SET/6 RED SPOTTY PAPER CUPS)   
43  (SET/6 RED SPOTTY PAPER PLATES, POSTAGE)    (SET/6 RED SPOTTY PAPER CUPS)   
35       (STRAWBERRY LUNCH BOX WITH CUTLERY)                        (POSTAGE)   
26      (ROUND SNACK BOXES SET OF4 WOODLAND)                        (POSTAGE)   
40             (SET/6 RED SPOTTY PAPER CUPS)  (SET/6 RED SPOTTY PAPER PLATES)   

    antecedent support  consequent support   support  confidence      lift  \
41            0.127551            0.137755  0.122449    0.960000  6.968889   
43            0.107143            0.137755  0.102041    0.952381  6.913580   
35            0.122449            0.765306  0.114796    0.937500  1.225000   
26            0.158163            0.765306  0.147959    0.935484  1.222366   
40            0.137755            0.127551  0.122449    0.888889  6.968889   

    representativity  leverage  conviction  

C:\Users\abhay\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
